# Model Verification

Structural and behavioral checks for simulation logic, including route flow and energy consumption mechanics.

Input data files are expected locally when rerunning this notebook.


https://www.sciencedirect.com/science/article/pii/S2666691X23000623#sec0012


In [ ]:
!pip install simpy

In [ ]:
import simpy
import numpy as np
import random

def format_time(minutes):
    hours = int(minutes // 60)
    mins = int(minutes % 60)
    return f"{hours:02d}:{mins:02d}"

class RouteSegment:
    def __init__(self, from_stop, to_stop, distance_km, slope_percent, from_name, to_name):
        self.from_stop = from_stop
        self.to_stop = to_stop
        self.distance_km = distance_km
        self.slope_percent = slope_percent
        self.from_name = from_name
        self.to_name = to_name

class Bus:
    def __init__(self, env, name, route_segments, depot_to_start, rounds, capacity, battery_capacity, co2_per_kwh, bus_area=22):
        self.env = env
        self.name = name
        self.route_segments = route_segments
        self.depot_to_start = depot_to_start
        self.rounds = rounds
        self.capacity = capacity
        self.bus_area = bus_area

        self.passengers = 0
        self.battery_capacity = battery_capacity
        self.battery_level = battery_capacity
        self.co2_per_kwh = co2_per_kwh

        self.total_co2_emissions = 0
        self.total_energy_consumed = 0
        self.passenger_counts = []
        self.routes_completed = 0
        self.total_passenger_transported = 0
        self.end_of_service_time = None

        self.action = env.process(self.run())

    def slope_correction(self, slope):
        if slope > 0:
            return 1 + (slope / 10)
        elif slope < 0:
            return max(0.7, 1 + (slope / 25))
        else:
            return 1.0

    def consumption_rate(self, passenger_mass, slope):
        base_consumptions = [0.8, 0.9, 1.0, 1.2, 1.4]
        mass_points = [0, 1000, 2000, 4000, 5700]
        base = np.interp(passenger_mass, mass_points, base_consumptions)
        slope_effect = 1 + (slope / 10)
        return max(base * slope_effect, 0.6)

    def travel_segment(self, segment, passenger_mass, log_passenger=True):
        travel_time = (segment.distance_km / 30) * 60
        yield self.env.timeout(travel_time)
        print(f'🛑 Bus {self.name} arriving at {segment.to_name} at {format_time(self.env.now)}')

        if log_passenger:
            leaving = random.randint(0, min(self.passengers, 10))
            self.passengers -= leaving
            possible_boarding = random.randint(5, 20)
            boarding = min(possible_boarding, self.capacity - self.passengers)
            self.passengers += boarding

            print(f'👋 {leaving} passengers off, 🚶 {boarding} boarded. Now {self.passengers} onboard.')
            passenger_mass = self.passengers * 70
            self.passenger_counts.append(self.passengers)

            # Toplam taşınan yolcu sayısını artır
            self.total_passenger_transported += boarding

            density = self.passengers / self.bus_area
            density_points = [0, 1, 2, 4, 6]
            boarding_times = [0.89, 1.13, 1.37, 1.67, 2.03]
            alighting_times = [0.59, 0.92, 1.11, 1.89, 5.92]

            boarding_time_per_pass = np.interp(density, density_points, boarding_times)
            alighting_time_per_pass = np.interp(density, density_points, alighting_times)

            boarding_total = boarding * boarding_time_per_pass
            alighting_total = leaving * alighting_time_per_pass

            waiting_time = max(boarding_total, alighting_total) + 8.8
            yield self.env.timeout(waiting_time / 60)
            print(f'⏱️ Waiting time: {waiting_time:.1f} sec (dynamic boarding/alighting)')

        consumption = self.consumption_rate(passenger_mass, segment.slope_percent) * segment.distance_km
        self.battery_level -= consumption
        self.total_energy_consumed += consumption
        co2_emission = consumption * self.co2_per_kwh
        self.total_co2_emissions += co2_emission

        print(f'🔋 Battery: {self.battery_level:.1f} kWh, 🌿 CO₂ emitted: {co2_emission:.2f} kg.')
        print(f'🟢 Bus {self.name} departing from {segment.to_name} at {format_time(self.env.now)}\n')

    def charge(self):
        print(f'⚡ Battery low at {self.battery_level:.1f} kWh. Charging at Edirnekapı Garajı ({format_time(self.env.now)})...')
        yield self.env.timeout(60)
        self.battery_level = self.battery_capacity * 0.8
        print(f'🔋 Charged to {self.battery_level:.1f} kWh at {format_time(self.env.now)}.\n')

    def run(self):
        print(f'🚌 Bus {self.name} departing from Edirnekapı Garajı at {format_time(self.env.now)}')
        yield from self.travel_segment(self.depot_to_start, passenger_mass=0, log_passenger=False)
        print(f'🛑 Bus {self.name} service starts at Topkapı at {format_time(self.env.now)}\n')

        for round_num in range(1, self.rounds + 1):
            print(f'🔁 Preparing round-trip {round_num} at Topkapı ({format_time(self.env.now)})')

            route_distance = sum(seg.distance_km for seg in self.route_segments) * 2
            avg_passengers = np.mean(self.passenger_counts) if self.passenger_counts else 30
            avg_mass = avg_passengers * 70
            avg_consumption = self.consumption_rate(avg_mass, 0)
            estimated_need = avg_consumption * route_distance + 50

            if self.battery_level < estimated_need:
                print(f'🚨 Not enough energy to start round-trip {round_num}. Returning to depot.')
                return_to_depot = RouteSegment(
                    self.depot_to_start.to_stop, self.depot_to_start.from_stop,
                    self.depot_to_start.distance_km, -self.depot_to_start.slope_percent,
                    self.depot_to_start.to_name, self.depot_to_start.from_name
                )
                yield from self.travel_segment(return_to_depot, passenger_mass=0, log_passenger=False)
                yield from self.charge()
                yield from self.travel_segment(self.depot_to_start, passenger_mass=0, log_passenger=False)

            print(f'🟢 Starting round-trip {round_num}\n')
            for segment in self.route_segments:
                yield from self.travel_segment(segment, passenger_mass=self.passengers * 70)

            print(f'🎯 Reached Beşiktaş Meydan. Starting return to Topkapı.')

            reversed_segments = [
                RouteSegment(s.to_stop, s.from_stop, s.distance_km, -s.slope_percent, s.to_name, s.from_name)
                for s in self.route_segments[::-1]
            ]
            for segment in reversed_segments:
                yield from self.travel_segment(segment, passenger_mass=self.passengers * 70)

            print(f'✅ Round-trip {round_num} completed at Topkapı ({format_time(self.env.now)})')
            print(f'👋 All {self.passengers} passengers disembark.\n')
            self.passengers = 0
            self.routes_completed += 1

        depot_return = RouteSegment(
            self.depot_to_start.to_stop, self.depot_to_start.from_stop,
            self.depot_to_start.distance_km, -self.depot_to_start.slope_percent,
            self.depot_to_start.to_name, self.depot_to_start.from_name
        )
        yield from self.travel_segment(depot_return, passenger_mass=0, log_passenger=False)
        self.end_of_service_time = self.env.now
        print(f'✅ Bus {self.name} returned to Edirnekapı Garajı and simulation ended at {format_time(self.env.now)}\n')

class Simulation:
    def __init__(self):
        self.env = simpy.Environment(initial_time=360)
        self.depot_to_start = RouteSegment(-1, 0, 2.1, -0.5, "Edirnekapı Garajı", "Topkapı")
        self.route_segments = [
            RouteSegment(i, i + 1, d, s, f, t)
            for i, (f, t, d, s) in enumerate([
                ("Topkapı", "Millet Caddesi", 1.265, -0.16),
                ("Millet Caddesi", "Pazartekke", 0.591, 0.51),
                ("Pazartekke", "İstanbul Tıp Fakültesi", 0.208, 1.44),
                ("İstanbul Tıp Fakültesi", "Çapa", 0.747, 0.13),
                ("Çapa", "Fındıkzade", 1.017, 1.08),
                ("Fındıkzade", "Haseki", 0.420, 3.57),
                ("Haseki", "Yusufpaşa", 0.468, 2.14),
                ("Yusufpaşa", "Pertevniyal Valide Sultan", 2.101, -0.24),
                ("Pertevniyal Valide Sultan", "İBB", 1.538, -0.91),
                ("İBB", "Vefa", 1.027, 0.58),
                ("Vefa", "Unkapanı", 0.401, 3.24),
                ("Unkapanı", "Haliç Metro", 1.571, 1.08),
                ("Haliç Metro", "Eminönü", 1.744, 0.11),
                ("Eminönü", "Karaköy", 0.853, -0.47),
                ("Karaköy", "Kemeraltı", 1.496, -0.53),
                ("Kemeraltı", "Tophane", 0.499, 1.00),
                ("Tophane", "Salıpazarı", 0.754, -0.80),
                ("Salıpazarı", "Fındıklı", 0.899, 0.33),
                ("Fındıklı", "Kabataş", 0.508, -0.39),
                ("Kabataş", "Mimar Sinan Üniversitesi", 1.046, 0.00),
                ("Mimar Sinan Üniversitesi", "Akaretler", 0.146, 2.05),
                ("Akaretler", "Beşiktaş Meydan", 0.459, -1.53),
            ])
        ]
        self.bus1 = Bus(self.env, "28T", self.route_segments, self.depot_to_start, rounds=6, capacity=90, battery_capacity=350, co2_per_kwh=0.233, bus_area=22)
        self.bus2 = Bus(self.env, "Karsan e-Atak", self.route_segments, self.depot_to_start, rounds=6, capacity=52, battery_capacity=220, co2_per_kwh=0.233, bus_area=17)

    def run(self):
        self.env.run(until=2000)
        for bus in [self.bus1, self.bus2]:
            route_km = sum(seg.distance_km for seg in self.route_segments)
            total_km = self.depot_to_start.distance_km + (route_km * 2 * bus.rounds) + self.depot_to_start.distance_km
            avg_passengers = np.mean(bus.passenger_counts) if bus.passenger_counts else 0
            avg_consumption = bus.total_energy_consumed / total_km if total_km else 0

            print(f"\n📊 {bus.name} Simulation Summary 📊")
            print(f"Routes Completed           : {bus.routes_completed}")
            print(f"Total Distance Travelled   : {total_km:.2f} km")
            print(f"Average Passengers         : {avg_passengers:.2f}")
            print(f"Total Energy Consumed      : {bus.total_energy_consumed:.2f} kWh")
            print(f"Average Consumption        : {avg_consumption:.2f} kWh/km")
            print(f"Total CO₂ Emitted          : {bus.total_co2_emissions:.2f} kg")
            print(f"Total Passengers Transported: {bus.total_passenger_transported}")
            print(f"End of Service             : {format_time(bus.end_of_service_time)}")

if __name__ == "__main__":
    sim = Simulation()
    sim.run()
